# 氷結晶粒径予測 - 学習・評価（Jupyter版）
`data/crops/` + `data/dataset.csv` を使って ResNet18 で回帰学習する。
`config.py` / `dataset.py` / `model.py` をそのまま利用するので、`vscode_project/` フォルダを
カレントディレクトリとしてこの notebook を実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import config
from dataset import CrystalDataset, load_valid_dataset, split_dataset, train_transform, test_transform
from model import build_model

config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")

## データ読み込み・分割

In [ ]:
df_all = load_valid_dataset(config.DATASET_CSV)
train_df, test_df = split_dataset(df_all)

train_df.to_csv(config.TRAIN_SPLIT_CSV, index=False, encoding="utf-8-sig")
test_df.to_csv(config.TEST_SPLIT_CSV, index=False, encoding="utf-8-sig")

train_loader = DataLoader(CrystalDataset(train_df, train_transform),
                           batch_size=config.BATCH_SIZE_TRAIN, shuffle=True,
                           num_workers=config.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(CrystalDataset(test_df, test_transform),
                          batch_size=config.BATCH_SIZE_TEST, shuffle=False,
                          num_workers=config.NUM_WORKERS, pin_memory=True)

## モデル準備

In [ ]:
model = build_model().to(device)
criterion = nn.HuberLoss(delta=1.0)
mse_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
print("モデル準備完了")

## 学習

In [ ]:
best_mse = float("inf")
history = []

for ep in range(1, config.EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    train_mse = total_loss / len(train_df)

    model.eval()
    t_mse = t_mae = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            out = model(x)
            t_mse += mse_fn(out, y).item() * x.size(0)
            t_mae += (out - y).abs().sum().item()
    test_mse = t_mse / len(test_df)
    test_mae = t_mae / len(test_df)

    scheduler.step(test_mse)
    tag = ""
    if test_mse < best_mse:
        best_mse = test_mse
        torch.save(model.state_dict(), config.MODEL_PATH)
        tag = "  ← best 保存"
    history.append({"epoch": ep, "train_mse": train_mse, "test_mse": test_mse, "test_mae": test_mae})
    print(f"Epoch {ep:2d}: train={train_mse:.4f}  test={test_mse:.4f}  MAE={test_mae:.4f} µm{tag}")

print(f"\n学習完了。best test MSE = {best_mse:.4f}")

## 学習曲線

In [ ]:
history_df = pd.DataFrame(history)
plt.plot(history_df["epoch"], history_df["train_mse"], label="train MSE")
plt.plot(history_df["epoch"], history_df["test_mse"], label="test MSE")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

## 評価（R²・MAE・RMSE）

In [ ]:
model.load_state_dict(torch.load(config.MODEL_PATH, map_location=device))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device)).squeeze(1).cpu().numpy()
        preds.extend(out.tolist())
        trues.extend(y.numpy().tolist())

preds = np.array(preds)
trues = np.array(trues)

ss_res = np.sum((trues - preds) ** 2)
ss_tot = np.sum((trues - trues.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
mae = np.mean(np.abs(trues - preds))
rmse = np.sqrt(np.mean((trues - preds) ** 2))

print(f"R²   = {r2:.4f}")
print(f"MAE  = {mae:.4f} µm")
print(f"RMSE = {rmse:.4f} µm")

plt.scatter(trues, preds, alpha=0.5)
lims = [min(trues.min(), preds.min()), max(trues.max(), preds.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("true (µm)")
plt.ylabel("predicted (µm)")
plt.title(f"R²={r2:.3f}")
plt.show()